In [ ]:
import time
import os
import pandas as pd
from seleniumbase import Driver
from bs4 import BeautifulSoup


AIRLINES = {
    "Qatar Airways": "Qatar-Airways",
    "Singapore Airlines": "Singapore-Airlines",
    "Cathay Pacific Airways": "Cathay-Pacific-Airways",
    "Emirates": "Emirates",
    "ANA All Nippon Airways": "ANA-All-Nippon-Airways",
    "Turkish Airlines": "Turkish-Airlines",
    "Korean Air": "Korean-Air",
    "Air France": "Air-France",
    "KLM Royal Dutch Airlines": "KLM-Royal-Dutch-Airlines",
    "Japan Airlines": "Japan-Airlines",
    "Hainan Airlines": "Hainan-Airlines",
    "Swiss International Air Lines": "Swiss-International-Air-Lines",
    "EVA Air": "EVA-Air",
    "British Airways": "British-Airways",
    "Qantas Airways": "Qantas-Airways",
    "Lufthansa": "Lufthansa",
    "Virgin Atlantic": "Virgin-Atlantic",
    "Saudia": "Saudia",
    "STARLUX Airlines": "STARLUX-Airlines",
    "Air Canada": "Air-Canada",
    "Iberia": "Iberia",
    "Delta Air Lines": "Delta-Air-Lines",
    "Austrian Airlines": "Austrian-Airlines",
    "Air New Zealand": "Air-New-Zealand",
    "Finnair": "Finnair",
    "Etihad Airways": "Etihad-Airways",
    "Malaysia Airlines": "Malaysia-Airlines",
    "AirAsia": "AirAsia",
    "Thai Airways": "Thai-Airways",
    "Scoot": "Scoot",
    "Fiji Airways": "Fiji-Airways",
    "Bangkok Airways": "Bangkok-Airways",
    "China Southern Airlines": "China-Southern-Airlines",
    "Virgin Australia": "Virgin-Australia",
    "Gulf Air": "Gulf-Air",
    "Air Astana": "Air-Astana",
    "China Airlines": "China-Airlines",
    "Ethiopian Airlines": "Ethiopian-Airlines",
    "IndiGo": "IndiGo",
    "Eurowings": "Eurowings",
    "Asiana Airlines": "Asiana-Airlines",
    "Vueling": "Vueling-Airlines",
    "LATAM Airlines": "LATAM-Airlines-Group",
    "Porter Airlines": "Porter-Airlines",
    "Volotea": "Volotea",
    "Garuda Indonesia": "Garuda-Indonesia",
    "Aegean Airlines": "Aegean-Airlines",
    "Oman Air": "Oman-Air",
    "Transavia France": "Transavia-France",
    "AZAL Azerbaijan Airlines": "AZAL-Azerbaijan-Airlines"
}

OUTPUT_FILE = "planespotters_fleet_matrix.csv"
CHECKPOINT_FILE = "planespotters_checkpoint.csv"

def scrape_data():
    all_data = []
    done_airlines = []
    
    if os.path.exists(CHECKPOINT_FILE):
        df_old = pd.read_csv(CHECKPOINT_FILE)
        all_data = df_old.to_dict(orient="records")
        done_airlines = df_old['airline_name'].unique().tolist()
        print(f"Resume: {len(done_airlines)} şirkət artıq var.")

    driver = Driver(browser="chrome", uc=True)
    
    try:
        for name, slug in AIRLINES.items():
            if name in done_airlines: continue
            
            url = f"https://www.planespotters.net/airline/{slug}"
            
            driver.uc_open_with_reconnect(url, reconnect_time=5)
            time.sleep(4)

            table = None
            retries = 2
            while retries > 0:
                soup = BeautifulSoup(driver.page_source, "html.parser")
                table = soup.find("table", class_="fleet_matrix")
                
                if table:
                    break
                
                print("Login, Captcha və ya Search səhifəsidirsə həll edin.")
                retries -= 1

            if not table:
                print(f"{name} tapıla bilmədi.")
                continue

            rows_found = 0
            for tr in table.find("tbody").find_all("tr"):
                th = tr.find("th")
                if not th or "subtype" in (th.get("class") or []) or "total" in th.text.strip().lower():
                    continue
                
                tds = tr.find_all("td")
                if len(tds) < 7: continue
                
                all_data.append({
                    "airline_name": name,
                    "aircraft_type": th.text.strip(),
                    "in_service": tds[0].text.strip(),
                    "parked": tds[1].text.strip(),
                    "total_current": tds[2].text.strip(),
                    "future": tds[3].text.strip(),
                    "historic": tds[4].text.strip(),
                    "avg_age": tds[5].text.strip(),
                    "total_all_time": tds[6].text.strip()
                })
                rows_found += 1
            
            print(f"{rows_found} sətir yığıldı.")
            
            pd.DataFrame(all_data).to_csv(CHECKPOINT_FILE, index=False, encoding="utf-8-sig")
            time.sleep(2)

        # --- FİNAL ---
        if all_data:
            df = pd.DataFrame(all_data)
            df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
            if os.path.exists(CHECKPOINT_FILE): os.remove(CHECKPOINT_FILE)
            print(f"\nFINISH! {OUTPUT_FILE} hazır!")

    finally:
        driver.quit()

if __name__ == "__main__":
    scrape_data()

Resume: 18 şirkət artıq var.

---> STARLUX Airlines oxunur...
Uğurlu: 3 sətir yığıldı.

---> Air Canada oxunur...
XƏBƏRDARLIQ: Air Canada üçün cədvəl hələ tapılmadı.
Lüfən browser-ə baxın: Login, Captcha və ya Search səhifəsidirsə həll edin.
XƏBƏRDARLIQ: Air Canada üçün cədvəl hələ tapılmadı.
Lüfən browser-ə baxın: Login, Captcha və ya Search səhifəsidirsə həll edin.
SKIPPED: Air Canada tapıla bilmədi.

---> Iberia oxunur...
Uğurlu: 17 sətir yığıldı.

---> Delta Air Lines oxunur...
Uğurlu: 22 sətir yığıldı.

---> Austrian Airlines oxunur...
Uğurlu: 17 sətir yığıldı.

---> Air New Zealand oxunur...
Uğurlu: 13 sətir yığıldı.

---> Finnair oxunur...
Uğurlu: 18 sətir yığıldı.

---> Etihad Airways oxunur...
Uğurlu: 14 sətir yığıldı.

---> Malaysia Airlines oxunur...
Uğurlu: 12 sətir yığıldı.

---> AirAsia oxunur...
Uğurlu: 6 sətir yığıldı.

---> Thai Airways oxunur...
Uğurlu: 3 sətir yığıldı.

---> Scoot oxunur...
Uğurlu: 6 sətir yığıldı.

---> Fiji Airways oxunur...
Uğurlu: 4 sətir yığıldı

In [2]:
df = pd.read_csv("planespotters_fleet_matrix.csv")

In [3]:
df

,airline_name,aircraft_type,in_service,parked,total_current,future,historic,avg_age,total_all_time
0,Qatar Airways,Airbus A300,NaN,NaN,NaN,NaN,10.0,NaN,10
1,Qatar Airways,Airbus A310,NaN,NaN,NaN,NaN,2.0,NaN,2
2,Qatar Airways,Airbus A319,NaN,NaN,NaN,NaN,3.0,NaN,3
3,Qatar Airways,Airbus A320,27.0,NaN,27.0,NaN,18.0,13.6 Years,45
4,Qatar Airways,Airbus A321,6.0,NaN,6.0,NaN,12.0,0.4 Years,18
...,...,...,...,...,...,...,...,...,...
608,AZAL Azerbaijan Airlines,Boeing 777,1.0,NaN,1.0,NaN,NaN,6.2 Years,1
609,AZAL Azerbaijan Airlines,Boeing 787 Dreamliner,2.0,NaN,2.0,1.0,NaN,11.3 Years,3
610,AZAL Azerbaijan Airlines,Canadair CL-44,NaN,NaN,NaN,NaN,1.0,NaN,1
611,AZAL Azerbaijan Airlines,Embraer ERJ-170,NaN,NaN,NaN,NaN,1.0,NaN,1


In [5]:
for i in df["aircraft_type"].unique():
    print(i)

Airbus A300
Airbus A310
Airbus A319
Airbus A320
Airbus A321
Airbus A330
Airbus A340
Airbus A350 XWB
Airbus A380
Boeing 737
Boeing 747
Boeing 757
Boeing 767
Boeing 777
Boeing 787 Dreamliner
Aérospatiale/BAC Concorde
McDonnell Douglas DC-10
Bristol 175 Britannia
Convair 880
Lockheed L-1011 TriStar
Lockheed L-188 Electra
De Havilland Canada DHC-8 Dash 8
Fokker 50 / 60
British Aerospace BAe 146 / Avro RJ
De Havilland Canada DHC-7 Dash 7
McDonnell Douglas DC-9
McDonnell Douglas MD-80
McDonnell Douglas MD-90
Airbus A220
Bombardier BD-700 Global
Fokker 70 / 100
McDonnell Douglas MD-11
ATR 42 / 72
Airbus A318
Bombardier CRJ-100 Series
Bombardier CRJ-700
Bombardier CRJ-900
Bombardier CRJ-1000
Convair 990
Dornier Do-328
Douglas DC-8
Embraer EMB-120 Brasilia
Embraer ERJ-145
Embraer ERJ-170
Embraer ERJ-190
Saab 340
Saab 2000
VFW-Fokker VFW 614
BAC 1-11
BAC Vickers VC-10
British Aerospace BAe ATP
British Aerospace Jetstream 31 / 32
British Aerospace Jetstream 41
Britten-Norman BN-2 Islander
De Havi